# 笔记本 08 — 投资组合优化与风控

**阶段 3 · 策略集成 (2 / 2)**

---

## 🎯 学习目标

| # | 目标 |
|---|------|
| 1 | 在遵守现金底线的前提下**归一化**集成权重 |
| 2 | 应用**仓位限制**（单一资产最大配置） |
| 3 | 实现制度相关的**现金底线**（牛市 20%、震荡 40%、熊市 50%） |
| 4 | 构建两级**熔断机制**（L1: 减仓, L2: 停止交易） |
| 5 | 理解**半凯利**仓位管理 |
| 6 | 从当前 → 目标权重生成**再平衡订单** |
| 7 | 与生产环境的 `PortfolioOptimizer`、`RiskManager`、`CircuitBreaker` 对比 |

### 前置要求
- NB07（集成输出 → `target_weights`）

In [ ]:
# ── 初始化 ─────────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True})
print("✅ 导入完毕  |  项目根目录:", ROOT)

---
## 1 · 风控管线

集成策略产出 `target_weights` 后，投资组合在执行前要通过**风控管线**：

```
ensemble_combine()
    │
    ▼
target_weights {sym: raw_weight}  +  cash_allocation
    │
    ├── 1. normalize_weights(cash_floor)
    ├── 2. enforce_position_limit(max_pct=0.10)
    ├── 3. circuit_breaker.evaluate(drawdown)
    │       ├── "ok"     → 继续
    │       ├── "reduce" → 权重减半
    │       └── "halt"   → 权重归零
    │
    ├── 4. 裁剪后重新归一化
    │
    └── generate_rebalance_orders(current, target, min_drift=0.15)
```

### 风控参数（来自 `config/strategy_params.yaml`）

```yaml
risk:
  max_position_pct: 0.10
  cash_floor_bull:    0.20
  cash_floor_ranging: 0.40
  cash_floor_bear:    0.50
  circuit_breaker:
    level_one: 0.03   # 3% 回撤 → 减仓
    level_two: 0.05   # 5% 回撤 → 停止交易
```

---
## 2 · 带现金底线的权重归一化

In [ ]:
def normalize_weights(weights: dict[str, float], cash_floor: float = 0.0) -> dict[str, float]:
    """缩放正权重使得总和 = (1 - cash_floor)。"""
    positive = {s: max(w, 0.0) for s, w in weights.items() if w > 0}
    total = sum(positive.values())
    if total <= 0 or cash_floor >= 1.0:
        return {}
    investable = 1.0 - max(cash_floor, 0.0)
    return {s: (w / total) * investable for s, w in positive.items()}

# 示例：来自 NB07 的原始集成输出
raw_weights = {
    "BTCUSDT":  0.145,
    "ETHUSDT":  0.263,
    "SOLUSDT":  0.225,
    "BNBUSDT":  0.035,
    "XRPUSDT":  0.015,
    "AVAXUSDT": 0.125,
}

print("原始权重 (总和={:.3f}):".format(sum(raw_weights.values())))
for s, w in raw_weights.items():
    print(f"  {s:12s}: {w:.4f}")

# 用牛市、震荡和熊市现金底线归一化
cash_floors = {"bull": 0.20, "ranging": 0.40, "bear": 0.50}
REGIME_NAMES = {"bull": "牛市", "ranging": "震荡", "bear": "熊市"}

print()
for regime, cf in cash_floors.items():
    normed = normalize_weights(raw_weights, cash_floor=cf)
    print(f"{REGIME_NAMES[regime]} (cash_floor={cf:.0%}, 可投资={1-cf:.0%}):")
    for s in sorted(normed, key=normed.get, reverse=True):
        print(f"  {s:12s}: {normed[s]:.4f}")
    print(f"  {'合计':12s}: {sum(normed.values()):.4f}  |  现金: {1-sum(normed.values()):.4f}\n")

In [ ]:
# 可视化：相同原始权重在不同现金底线下的呈现
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
bar_colours = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c"]

for ax, (regime, cf) in zip(axes, cash_floors.items()):
    normed = normalize_weights(raw_weights, cash_floor=cf)
    syms = sorted(normed.keys())
    vals = [normed[s] for s in syms]
    ax.bar(syms, vals, color=bar_colours[:len(syms)], alpha=0.8)
    ax.bar(["现金"], [1- sum(vals)], color="#95a5a6", alpha=0.6)
    ax.set_title(f"{REGIME_NAMES[regime]} (cash_floor={cf:.0%})")
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylim(0, 0.55)

axes[0].set_ylabel("权重")
fig.suptitle("按制度现金底线的权重归一化", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3 · 仓位限制

单一资产不应超过投资组合的 **10%** — 这是 `max_position_pct` 的保护机制。

$$
w_i^{\text{clipped}} = \min(w_i, 0.10)
$$

裁剪后需要**重新归一化**，因为总和可能低于可投资比例。

In [ ]:
def enforce_position_limit(weights: dict[str, float], max_pct: float) -> dict[str, float]:
    """将每个权重裁剪到 max_pct。"""
    return {s: min(max(w, 0.0), max_pct) for s, w in weights.items() if w > 0}

MAX_POS = 0.10

# 取牛市归一化后的权重
bull_normed = normalize_weights(raw_weights, cash_floor=0.20)
clipped = enforce_position_limit(bull_normed, MAX_POS)

print("裁剪前 → 裁剪后:")
for s in sorted(bull_normed, key=bull_normed.get, reverse=True):
    before = bull_normed[s]
    after = clipped[s]
    flag = " ✂️ 已裁剪" if after < before else ""
    print(f"  {s:12s}: {before:.4f} → {after:.4f}{flag}")

print(f"\n裁剪前合计: {sum(bull_normed.values()):.4f}")
print(f"裁剪后合计: {sum(clipped.values()):.4f}")
print(f"损失权重:   {sum(bull_normed.values()) - sum(clipped.values()):.4f}")

In [ ]:
# 裁剪后重新归一化以填充可投资比例
def full_risk_pipeline(raw_weights, cash_floor, max_pos):
    """归一化 → 裁剪 → 重新归一化。"""
    step1 = normalize_weights(raw_weights, cash_floor)
    step2 = enforce_position_limit(step1, max_pos)
    step3 = normalize_weights(step2, cash_floor)  # 重新归一化
    return step1, step2, step3

s1, s2, s3 = full_risk_pipeline(raw_weights, 0.20, 0.10)

print(f"{'步骤':6s} {'总和':>8s}  {'最大值':>8s}")
print(f"{'归一化':6s} {sum(s1.values()):8.4f}  {max(s1.values()):8.4f}")
print(f"{'裁剪':6s} {sum(s2.values()):8.4f}  {max(s2.values()):8.4f}")
print(f"{'重归一':6s} {sum(s3.values()):8.4f}  {max(s3.values()):8.4f}")

> **注意：** 重新归一化后，如果资产数量较少，某些权重可能再次超过 `max_pos`。在生产环境中，我们会迭代
直到所有权重都低于限制。

---
## 4 · 熔断机制

熔断机制监控**投资组合回撤**并采取保护措施：

| 级别 | 回撤 | 操作 |
|------|------|------|
| OK | < 3% | 正常交易 |
| **L1** | 3% – 5% | `reduce` — 所有仓位权重减半 |
| **L2** | ≥ 5% | `halt` — 所有权重归零（全现金） |

In [ ]:
class CircuitBreaker:
    def __init__(self, level_one: float = 0.03, level_two: float = 0.05):
        self.level_one = level_one
        self.level_two = level_two
    
    def evaluate(self, drawdown_pct: float) -> str:
        if drawdown_pct >= self.level_two:
            return "halt"
        if drawdown_pct >= self.level_one:
            return "reduce"
        return "ok"

cb = CircuitBreaker(level_one=0.03, level_two=0.05)

# 遍历回撤值
drawdowns = np.arange(0, 0.08, 0.005)
actions = [cb.evaluate(dd) for dd in drawdowns]
action_colours = {"ok": "#2ecc71", "reduce": "#f39c12", "halt": "#e74c3c"}
action_names = {"ok": "正常", "reduce": "减仓", "halt": "停止"}

fig, ax = plt.subplots(figsize=(12, 3))
for i, (dd, action) in enumerate(zip(drawdowns, actions)):
    ax.bar(i, 1, color=action_colours[action], width=0.9, alpha=0.7)
    ax.text(i, 0.5, action_names[action], ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_xticks(range(len(drawdowns)))
ax.set_xticklabels([f"{dd:.1%}" for dd in drawdowns], rotation=45)
ax.set_xlabel("投资组合回撤")
ax.set_yticks([])
ax.set_title("熔断机制响应")
plt.tight_layout()
plt.show()

In [ ]:
def apply_circuit_breaker(weights: dict[str, float], drawdown: float) -> dict[str, float]:
    """对投资组合权重应用熔断机制。"""
    action = cb.evaluate(drawdown)
    if action == "halt":
        return {s: 0.0 for s in weights}  # 全现金
    elif action == "reduce":
        return {s: w * 0.5 for s, w in weights.items()}  # 敞口减半
    return weights  # 不变

# 示例：在不同回撤水平下的投资组合
final_weights = s3  # 来自 full_risk_pipeline

for dd in [0.01, 0.035, 0.06]:
    adjusted = apply_circuit_breaker(final_weights, dd)
    action = cb.evaluate(dd)
    total = sum(adjusted.values())
    print(f"回撤={dd:.1%}  操作={action_names[action]:4s}  股权总计={total:.4f}  现金={1-total:.4f}")

---
## 5 · 模拟回撤与熔断机制影响

In [ ]:
# 模拟触发熔断机制的净值曲线
np.random.seed(99)
n_days = 100
daily_returns = np.concatenate([
    np.random.normal(0.002, 0.012, 40),  # 正常期
    np.random.normal(-0.005, 0.020, 30), # 回撤期
    np.random.normal(0.003, 0.010, 30),  # 恢复期
])

# 无熔断
equity_no_cb = [1.0]
for r in daily_returns:
    equity_no_cb.append(equity_no_cb[-1] * (1 + r))
equity_no_cb = np.array(equity_no_cb)

# 有熔断
equity_with_cb = [1.0]
cb_actions = ["ok"]
for i, r in enumerate(daily_returns):
    peak = max(equity_with_cb)
    current = equity_with_cb[-1]
    dd = (peak - current) / peak if peak > 0 else 0
    
    action = cb.evaluate(dd)
    cb_actions.append(action)
    
    if action == "halt":
        effective_r = 0.0
    elif action == "reduce":
        effective_r = r * 0.5
    else:
        effective_r = r
    
    equity_with_cb.append(equity_with_cb[-1] * (1 + effective_r))

equity_with_cb = np.array(equity_with_cb)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(equity_no_cb, label="无熔断", lw=1.5, alpha=0.8, color="#e74c3c")
ax1.plot(equity_with_cb, label="有熔断", lw=1.5, alpha=0.8, color="#2ecc71")
ax1.set_ylabel("净值")
ax1.set_title("熔断机制对净值曲线的影响")
ax1.legend()

# 无保护投资组合的回撤
peak_eq = np.maximum.accumulate(equity_no_cb)
drawdown = (peak_eq - equity_no_cb) / peak_eq
ax2.fill_between(range(len(drawdown)), drawdown, alpha=0.4, color="#e74c3c")
ax2.axhline(0.03, ls="--", color="#f39c12", lw=1, label="L1 (3%)")
ax2.axhline(0.05, ls="--", color="#e74c3c", lw=1, label="L2 (5%)")
ax2.set_ylabel("回撤")
ax2.set_xlabel("天")
ax2.legend(fontsize=8)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

max_dd_no_cb = drawdown.max()
peak_cb = np.maximum.accumulate(equity_with_cb)
max_dd_with_cb = ((peak_cb - equity_with_cb) / peak_cb).max()
print(f"无熔断最大回撤: {max_dd_no_cb:.2%}")
print(f"有熔断最大回撤: {max_dd_with_cb:.2%}")
print(f"无熔断最终净值: {equity_no_cb[-1]:.4f}")
print(f"有熔断最终净值: {equity_with_cb[-1]:.4f}")

---
## 6 · 半凯利仓位管理

**凯利公式**给出最佳下注比例：

$$
f^* = \frac{p}{a} - \frac{q}{b}
$$

其中 $p$ = 胜率, $q = 1-p$, $a$ = 平均亏损, $b$ = 平均盈利。

实践中使用**半凯利** ($f^*/2$) 因为：
- 凯利假设完美知道 $p$ 和 $b/a$
- 真实分布有肥尾
- 半凯利达到约 75% 的凯利增长率，方差仅为约 50%

In [ ]:
def kelly_fraction(win_prob: float, avg_win: float, avg_loss: float) -> float:
    """完整凯利比例。"""
    q = 1 - win_prob
    if avg_loss == 0 or avg_win == 0:
        return 0.0
    return win_prob / avg_loss - q / avg_win

def half_kelly(win_prob: float, avg_win: float, avg_loss: float) -> float:
    """半凯利：更安全、更实用。"""
    return max(0.0, kelly_fraction(win_prob, avg_win, avg_loss) / 2)

# 示例：我们的机器人历史胜率 55%，平均盈利 2.5%，平均亏损 1.8%
p = 0.55
avg_w = 0.025
avg_l = 0.018

fk = kelly_fraction(p, avg_w, avg_l)
fhk = half_kelly(p, avg_w, avg_l)
print(f"胜率:        {p:.0%}")
print(f"平均盈利:    {avg_w:.1%}")
print(f"平均亏损:    {avg_l:.1%}")
print(f"完整凯利:    {fk:.1%}")
print(f"半凯利:      {fhk:.1%}")

In [ ]:
# 敏感性：凯利比例 vs 胜率
win_rates = np.arange(0.40, 0.75, 0.01)
full_kellys = [kelly_fraction(p, avg_w, avg_l) for p in win_rates]
half_kellys = [half_kelly(p, avg_w, avg_l) for p in win_rates]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(win_rates, full_kellys, label="完整凯利", lw=2, color="#e74c3c")
ax.plot(win_rates, half_kellys, label="半凯利", lw=2, color="#2ecc71")
ax.axhline(0.10, ls="--", color="gray", lw=0.8, label="仓位上限 (10%)")
ax.axhline(0, ls="-", color="black", lw=0.5)
ax.axvline(0.55, ls=":", color="#3498db", lw=1, label="我们的胜率 (55%)")
ax.set_xlabel("胜率")
ax.set_ylabel("最优比例")
ax.set_title("凯利公式 vs 胜率")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7 · 再平衡订单生成

得到最终目标权重后，我们将其与**当前**投资组合权重对比，生成订单。只有变化超过 `min_drift=0.15`（即 15 个百分点）的才触发交易。

In [ ]:
def generate_rebalance_orders(
    current: dict[str, float],
    target: dict[str, float],
    min_drift: float = 0.0,
) -> list[dict]:
    """为权重差异显著的币种生成买入/卖出提案。"""
    orders = []
    all_symbols = set(current) | set(target)
    for sym in sorted(all_symbols):
        t = target.get(sym, 0.0)
        c = current.get(sym, 0.0)
        drift = t - c
        if abs(drift) <= min_drift:
            continue
        orders.append({
            "side": "BUY" if drift > 0 else "SELL",
            "symbol": sym,
            "target_weight": t,
            "current_weight": c,
            "drift": drift,
        })
    return orders

# 当前投资组合（上次再平衡后）
current_weights = {
    "BTCUSDT": 0.10,
    "ETHUSDT": 0.10,
    "SOLUSDT": 0.05,
    "BNBUSDT": 0.10,
    "XRPUSDT": 0.10,
    "AVAXUSDT": 0.05,
}

# 新目标（经过完整风控管线后）
target = s3

# 生成订单
MIN_DRIFT = 0.02  # 演示用 2% 阈值
orders = generate_rebalance_orders(current_weights, target, min_drift=MIN_DRIFT)

print(f"再平衡订单 (min_drift={MIN_DRIFT:.0%}):")
print(f"{'方向':<5} {'币种':<12} {'当前':>8} {'目标':>8} {'偏移':>8}")
print("-" * 45)
for o in orders:
    print(f"{o['side']:<5} {o['symbol']:<12} {o['current_weight']:8.4f} {o['target_weight']:8.4f} {o['drift']:+8.4f}")

In [ ]:
# 可视化当前 vs 目标
all_syms = sorted(set(current_weights) | set(target))
x = np.arange(len(all_syms))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, [current_weights.get(s, 0) for s in all_syms], w, label="当前", color="#3498db", alpha=0.7)
ax.bar(x + w/2, [target.get(s, 0) for s in all_syms], w, label="目标", color="#e74c3c", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(all_syms, rotation=45)
ax.set_ylabel("权重")
ax.set_title("当前 vs 目标 投资组合权重")
ax.legend()
ax.axhline(0.10, ls="--", color="gray", lw=0.8, label="仓位上限")
plt.tight_layout()
plt.show()

---
## 8 · 生产代码对比

In [ ]:
from bot.strategy.portfolio_optimizer import PortfolioOptimizer, normalize_weights as prod_normalize
from bot.risk.risk_manager import RiskManager, enforce_position_limit as prod_enforce
from bot.risk.circuit_breaker import CircuitBreaker as ProdCB
from bot.execution.order_executor import generate_rebalance_orders as prod_gen_orders

# 1. 归一化
po = PortfolioOptimizer(cash_floor=0.20)
prod_normed = po.optimize(raw_weights)
our_normed = normalize_weights(raw_weights, 0.20)
match_norm = all(abs(prod_normed.get(s, 0) - our_normed.get(s, 0)) < 1e-9 for s in set(prod_normed) | set(our_normed))
print(f"normalize_weights 一致: {'✅' if match_norm else '❌'}")

# 2. 仓位限制
rm = RiskManager(max_position_pct=0.10)
prod_clipped = rm.apply_position_limits(prod_normed)
our_clipped = enforce_position_limit(our_normed, 0.10)
match_clip = all(abs(prod_clipped.get(s, 0) - our_clipped.get(s, 0)) < 1e-9 for s in set(prod_clipped) | set(our_clipped))
print(f"enforce_position_limit 一致: {'✅' if match_clip else '❌'}")

# 3. 熔断机制
pcb = ProdCB(level_one=0.03, level_two=0.05)
for dd in [0.01, 0.035, 0.06]:
    ours = cb.evaluate(dd)
    prod = pcb.evaluate(dd)
    print(f"CB dd={dd:.1%}: 手写={ours:6s} 生产={prod:6s} {'✅' if ours == prod else '❌'}")

---
## 9 · 完整风控管线总结

```
集成 target_weights
        │
        ▼
   ┌────────────────────────┐
   │ normalize_weights()    │ → 总和 = 1 - cash_floor
   │ (按制度选择现金底线)   │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ enforce_position_limit │ → 每个裁剪到 10%
   │ (max_position=0.10)    │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ circuit_breaker.eval() │ → ok / reduce / halt
   │ (L1=3%, L2=5%)        │
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ 重新归一化             │ → 确保总和正确
   └────────┬───────────────┘
            ▼
   ┌────────────────────────┐
   │ generate_rebalance_    │ → 买入 / 卖出 订单
   │ orders(min_drift=0.15) │
   └────────────────────────┘
```

---
## 🔬 练习

1. **迭代裁剪：** 实现一个循环，裁剪 → 重新归一化直到所有权重 ≤ `max_pos`。3 个资产时需要几次迭代？

2. **动态熔断：** 根据制度修改 L1/L2 阈值：牛市(4%/7%)、震荡(3%/5%)、熊市(2%/3%)。这如何影响回撤？

3. **凯利 + 制度：** 使用模拟胜率计算每种策略每种制度的凯利比例。哪个制度-策略对的凯利比例最高？

4. **交易成本：** 给再平衡订单生成器添加 0.1% 的手续费。什么最小偏移量能使成本拖累最优化？

---
## ✅ 知识检查

1. 为什么在应用仓位限制*之前*先归一化权重？
2. 仓位裁剪后未分配的权重去了哪里？
3. 为什么 L2（停止）比 L1（减仓）更激进？什么时候会从 L2 恢复？
4. 为什么用半凯利而不是完整凯利？
5. `min_drift` 是什么？它为什么存在？

---
## 🔗 下一步

**[NB09 — 回测引擎 →](09_回测引擎.ipynb)**

我们将构建一个前向优化回测器，在历史数据上测试整个管线，计算夏普、索提诺、卡尔马、收益因子和胜率。